# 04-2. RF-DETR weight로 Label Studio prediction 생성

이 notebook은 이미 Label Studio project에 import된 image task를 대상으로 RF-DETR custom weight를 실행하고, 결과 bbox를 Label Studio의 `prediction`으로 업로드합니다.

안전 기본값은 `DRY_RUN=True`입니다. 이 상태에서는 RF-DETR model을 로드하거나 inference/upload를 하지 않고, project/task/image 경로와 class 설정만 확인합니다.

핵심 용어:
- `MODEL_WEIGHTS`: RF-DETR 학습 결과 checkpoint 경로입니다. 보통 `checkpoint_best_total.pth`처럼 생겼습니다.
- `MODEL_VARIANT`: RF-DETR 모델 크기입니다. `"nano"`, `"small"`, `"medium"`, `"large"`, `"auto"` 중 선택합니다.
- `CLASS_YAML`: 학습 때 사용한 YOLO dataset yaml입니다. `names` 순서가 RF-DETR 학습 class order와 반드시 같아야 합니다.
- `CONF`: RF-DETR `predict(..., threshold=...)`에 들어가는 기본 confidence threshold입니다.
- `CLASS_THRESH`: class별 confidence threshold입니다. 낮은 class별 threshold를 쓰려면 `CONF`도 충분히 낮게 두는 것이 안전합니다.
- `IOU`, `CLASS_IOU`: RF-DETR 결과에 대해 class별 NMS를 한 번 더 적용할 때 쓰는 값입니다. RF-DETR 자체 후처리와 중복될 수 있으므로 중복 bbox가 많을 때만 조정하세요.
- `META_TAG`, `IMPORT_ID`, `PRED_MODEL`: 나중에 merge/delete/export에서 결과를 구분하기 위한 이름표입니다.


In [ ]:
from pathlib import Path
import sys

# notebook을 repo 어느 위치에서 열어도 src package를 찾기 위한 bootstrap입니다.
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from labelstudio_bbox_tools.config import settings_from_env

settings = settings_from_env(REPO_ROOT / ".env")
print("repo:", REPO_ROOT)
print("Label Studio URL:", settings.url)
print("doc_root:", settings.doc_root)


In [ ]:
from pathlib import Path
from labelstudio_bbox_tools.pseudo_label.rfdetr import auto_label_rfdetr, load_rfdetr_class_names

# ===== 사용자 설정 =====
PROJECT_ID = 123

# RF-DETR 학습 결과 checkpoint 경로입니다. 직접 사용하는 checkpoint_best_total.pth 경로로 바꾸세요.
MODEL_WEIGHTS = Path("/path/to/checkpoint_best_total.pth")

# checkpoint가 model_name을 담고 있으면 "auto"가 가장 안전합니다.
# 직접 variant를 고르고 싶으면 "nano", "small", "medium", "large" 중 하나를 사용하세요.
MODEL_VARIANT = "medium"

# 권장: 학습 때 사용한 dataset yaml의 names 필드에서 class 순서를 자동으로 읽습니다.
# RF-DETR 학습 class order와 반드시 같은 yaml을 사용하세요.
CLASS_YAML = Path("/path/to/data.yaml")

# fallback: yaml이 없을 때만 수기로 class 순서를 적습니다.
MANUAL_CLASSES = []

# RF-DETR는 device를 모델 config로 받습니다. 보통 "cuda" 또는 "cpu"입니다.
DEVICE = "cuda"

BATCH_SIZE = 1  # RF-DETR predict는 image별 호출로 처리합니다. 안정성을 위해 1부터 권장합니다.
CONF = 0.30
IOU = 0.60

# class별 confidence threshold입니다. 없는 class는 default 또는 CONF를 사용합니다.
CLASS_THRESH = {
    "default": CONF,
}

# class별 2차 NMS IoU threshold입니다. RF-DETR 자체 후처리와 중복될 수 있습니다.
CLASS_IOU = {
    "default": IOU,
}

META_TAG = "my_rfdetr_pseudo_label_run"
IMPORT_ID = "my_import_id"
PRED_MODEL = META_TAG

SAVE_LOCAL = False
LOCAL_OUT_DIR = REPO_ROOT / "outputs" / "rfdetr_pseudo_label_preview"
SAVE_FORMAT = "json"  # "json" 또는 "yolo"

MAX_TASKS = None      # 처음 테스트할 때는 작은 값으로 확인하세요.
DRY_RUN = True        # False로 바꾸면 실제 RF-DETR inference와 prediction 업로드를 수행합니다.


In [ ]:
# class 목록 확인 cell입니다.
# CLASS_YAML 경로가 아직 placeholder라면 에러가 납니다. 그 경우 CLASS_YAML=None으로 두고 MANUAL_CLASSES를 채우세요.
class_yaml = CLASS_YAML if CLASS_YAML and CLASS_YAML.exists() else None
classes = load_rfdetr_class_names(class_yaml=class_yaml, manual_classes=MANUAL_CLASSES)
print("class count:", len(classes))
print(classes)


In [ ]:
summary = auto_label_rfdetr(
    ls_url=settings.url,
    api_key=settings.api_key,
    project_id=PROJECT_ID,
    model_weights=MODEL_WEIGHTS,
    doc_root=settings.doc_root,
    class_yaml=class_yaml,
    classes=MANUAL_CLASSES,
    model_variant=MODEL_VARIANT,
    device=DEVICE,
    conf=CONF,
    iou=IOU,
    batch_size=BATCH_SIZE,
    class_conf_map=CLASS_THRESH,
    class_iou_map=CLASS_IOU,
    save_local=SAVE_LOCAL,
    local_out_dir=LOCAL_OUT_DIR,
    save_format=SAVE_FORMAT,
    pred_model=PRED_MODEL,
    import_id=IMPORT_ID,
    meta_tag=META_TAG,
    dry_run=DRY_RUN,
    max_tasks=MAX_TASKS,
)
summary.as_dict()
